# Gemma 2 9B Fine-tuning for Pet Counseling

이 노트북은 Gemma 2 9B 모델을 '다정하고 전문적인 펫 상담사'가 되도록 파인튜닝하는 과정을 담고 있습니다.

### 1. 준비 사항
- **런타임 유형**: [수정] -> [노트북 설정] -> [하드웨어 가속기] -> **T4 GPU** 혹은 **L4 GPU** 선택
- **Hugging Face 토큰**: [Hugging Face Settings](https://huggingface.co/settings/tokens)에서 토큰(Write 권한)을 생성하세요.
- **데이터셋**: 왼쪽 파일(폴더 아이콘) 탭에 `train_dataset.jsonl` 파일을 업로드하세요.

In [1]:
# 1. 필수 라이브러리 설치
!pip install -q -U transformers datasets peft trl bitsandbytes accelerate python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.4 MB/s eta 0:00:00


In [2]:
# 2. 임포트 및 로그인
import os
import torch
from huggingface_hub import login
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig
from trl import SFTTrainer

# 코랩 Secrets 활용 또는 직접 입력
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except:
    hf_token = input("Hugging Face Write Token을 입력하세요: ")

login(token=hf_token)

In [3]:
# 3. 데이터셋 로드 및 포맷팅
dataset_path = "train_dataset.jsonl"
dataset = load_dataset("json", data_files=dataset_path, split="train")

def format_gemma_prompt(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = (
            f"<start_of_turn>user\n"
            f"{instruction}\n\n"
            f"질문: {input_text}<end_of_turn>\n"
            f"<start_of_turn>model\n"
            f"{output}<end_of_turn>"
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(format_gemma_prompt, batched=True)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2787 [00:00<?, ? examples/s]

In [4]:
# 4. 모델 및 토크나이저 설정
model_id = "google/gemma-2-9b-it"
output_dir = "./gemma2-pet-counselor-lora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 98.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 9.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.41 GiB is allocated by PyTorch, and 23.65 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# 5. 파인튜닝 설정 및 실행
peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.1,
    r=16,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    optim="paged_adamw_32bit",
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="constant",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=False,
)

print("Training Started!")
trainer.train()

trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")